In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow import log_metric, log_param, log_artifact

## Load models

In [2]:
import tensorflow as tf
from pathlib import Path

# Get all models with endswith .pkl and .keras from pipelines/models directory
import os
base_dir = Path.cwd().resolve().parents[3]
model_dir = base_dir / "pipelines" / "models"
model_files = [f for f in os.listdir(model_dir) if f.endswith(".pkl") or f.endswith(".keras")]
print("Model files found:", model_files)

Model files found: ['lstm_model.keras', 'scaler_X.pkl', 'scaler_profit.pkl', 'scaler_sales.pkl']


## Generate to mlflow

In [15]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.openai.autolog()
mlflow.groq.autolog()

In [31]:
import os
from dotenv import load_dotenv
from pathlib import Path
from groq import Groq
import mlflow
from IPython.display import display, HTML

# Load environment variables
env_path = Path.cwd().resolve().parents[3] / ".env"
load_dotenv(dotenv_path=env_path)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

# ✅ Set tracking URI
mlflow.set_tracking_uri("http://localhost:5007")

try:
    mlflow.set_experiment("groq_inference")
    
    with mlflow.start_run(run_name="groq_test"):
        client = Groq()
        message = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": "What is the capital of France?"}],
            temperature=0.7
        )
        
        response_text = message.choices[0].message.content
        token_count = len(response_text.split())
        
        mlflow.log_param("model", "llama-3.3-70b-versatile")
        mlflow.log_param("temperature", 0.7)
        mlflow.log_metric("token_count", token_count)
        
        display(HTML(f"""
        <div style='border:2px solid #4CAF50;padding:15px;border-radius:10px'>
            <h3>✅ MLflow Run Logged</h3>
            <p><b>Status:</b> Successfully logged to MLflow server</p>
            <p><b>Response:</b> {response_text}</p>
            <p><b>View Dashboard:</b> <a href='http://localhost:5007' target='_blank'>http://localhost:5007</a></p>
        </div>
        """))
        
except Exception as e:
    display(HTML(f"<div style='color:red;padding:10px'>❌ Error: {str(e)}<br>Make sure MLflow server is running on port 5007</div>"))

2026/03/16 23:35:30 INFO mlflow.tracking.fluent: Experiment with name 'groq_inference' does not exist. Creating a new experiment.


🏃 View run groq_test at: http://localhost:5007/#/experiments/219342459421682435/runs/1f4322d080134805be0a8dc23b297113
🧪 View experiment at: http://localhost:5007/#/experiments/219342459421682435


Trace(trace_id=tr-db604d3a015f6e8a7c2de08a667ff44e)